# THS-II EMS - RL vs Fixed-Mode Benchmark (Colab)

Train and compare **reinforcement-learning energy-management agents** (PPO, A2C, DQN, QR-DQN)
against the **fixed drive modes** (EV / ECO / NORMAL / PWR) and a **rule-based baseline**,
all run through the *same* `THSEnv` Toyota-Prius hybrid powertrain simulator so every number
is directly comparable.

The agent picks one of 4 discrete drive modes each step; reward is charge-sustaining
(minimise fuel while holding SOC near 60%).

---

## What to upload to Colab

This notebook needs the project's environment + powertrain model. Two options:

**Option A - one zip (recommended).** Upload **`ths_colab_bundle.zip`** (created next to this
notebook). It contains exactly:
```
modeling.py
env/__init__.py
env/ths_env.py
env/drive_cycles/{WLTC,FTP75,US06}.csv
env/drive_cycles/{GENERAL,GENERAL_phases}.csv
eval/benchmark_general.py
```
Run the *Setup* cell below - it unzips and puts everything on the path.

**Option B - pretrained model (optional).** If you also upload your existing
`best_model.zip`, the benchmark will include it as **`PPO-pretrained`** without retraining.

**Option C - bring your own trained agents (no retraining).** Already trained your
PPO / A2C / DQN / QR-DQN agents elsewhere? Skip Section 4 entirely. Run the
**"Use models you already trained"** cell (just after the sanity check), upload your
`.zip` checkpoints, and they get loaded straight into the benchmarks - both the
WLTC/FTP75/US06 comparison (Sections 5-7) and the **GENERAL all-regime test**
(Section 9, from `eval/benchmark_general.py`: CO2, total energy, SOC trace, fuel,
battery-life prediction, and per-road-part mode decisions).

> Tip: set the runtime to **GPU** (Runtime -> Change runtime type) - training is faster.


## 1 - Install dependencies

In [ ]:
# Colab ships torch already; we pin the SB3 stack the project uses.
!pip -q install "stable-baselines3==2.8.0" "sb3-contrib==2.8.0" "gymnasium==0.29.1" 2>/dev/null
import stable_baselines3, gymnasium, sb3_contrib
print("stable-baselines3", stable_baselines3.__version__)
print("gymnasium       ", gymnasium.__version__)
print("sb3-contrib     ", sb3_contrib.__version__)

## 2 - Setup: upload & unpack the project bundle

Run this cell, then pick `ths_colab_bundle.zip` (Option A) when prompted. If you've already
placed the files in the Colab filesystem some other way, the cell will detect them and skip
the upload.

In [ ]:
import os, sys, zipfile
from pathlib import Path

PROJECT_ROOT = Path("/content/ths_project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

def have_project():
    return (PROJECT_ROOT / "modeling.py").exists() and (PROJECT_ROOT / "env" / "ths_env.py").exists()

if not have_project():
    try:
        from google.colab import files  # type: ignore
        print("Upload ths_colab_bundle.zip ...")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(PROJECT_ROOT)
                print("Extracted", name)
    except ImportError:
        print("Not on Colab - assuming files are already present.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert have_project(), "modeling.py / env/ths_env.py not found. Upload ths_colab_bundle.zip."
print("\nProject files:")
for p in ["modeling.py", "env/ths_env.py", "env/drive_cycles/WLTC.csv",
          "env/drive_cycles/FTP75.csv", "env/drive_cycles/US06.csv",
          "env/drive_cycles/GENERAL.csv", "env/drive_cycles/GENERAL_phases.csv",
          "eval/benchmark_general.py"]:
    print(f"  {'OK ' if (PROJECT_ROOT / p).exists() else 'MISSING'} {p}")

## 3 - Sanity check: build the env and step it once

In [ ]:
import numpy as np
from env.ths_env import THSEnv

env = THSEnv(cycle="WLTC")
obs, _ = env.reset(seed=0)
print("observation space:", env.observation_space)
print("action space     :", env.action_space, "->", [m.name for m in THSEnv.ACTION_TO_MODE])
obs, r, done, trunc, info = env.step(2)   # NORMAL
print("first step reward :", round(r, 3))
print("episode length    :", len(env.profile), "steps")
print("sanity check passed")

## Use models you already trained (skip Section 4)

Already trained your agents? **Run this cell instead of Section 4** and upload either:

* **a single SB3 checkpoint** (`best_model.zip`, `final.zip`, ...) - the algorithm is
  auto-detected from the filename (`ppo`/`a2c`/`dqn`/`qrdqn`; use `ALGO_OVERRIDE` if not), or
* **the `ths_trained_models.zip` bundle** that the training notebook produces - it holds
  `PPO/best_model.zip`, `A2C/best_model.zip`, `DQN/...`, `QRDQN/...`; the cell unpacks it and
  registers every algorithm automatically.

You can mix and select several files at once. Every model is loaded once and reused by
**both** the WLTC/FTP75/US06 benchmark (Sections 5-7) and the GENERAL test (Section 9) -
no retraining.


In [ ]:
# === Use models you already trained (no retraining) =========================
# Accepts a single SB3 .zip checkpoint OR the ths_trained_models.zip bundle
# (PPO/best_model.zip, A2C/best_model.zip, ...) from the training notebook.
import shutil, zipfile
from pathlib import Path
import torch
from stable_baselines3 import PPO, A2C, DQN
from sb3_contrib import QRDQN

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_ALGO_CLASS = {"PPO": PPO, "A2C": A2C, "DQN": DQN, "QRDQN": QRDQN}

# Force the algorithm for a filename, e.g. {"my_run.zip": "QRDQN"}
ALGO_OVERRIDE = {}
# Give a nicer label to a single-model file, e.g. {"best_model.zip": "PPO-pretrained"}
LABEL_OVERRIDE = {}

def _guess_algo(name):
    s = str(name).lower()
    if "qrdqn" in s or "qr-dqn" in s or "qr_dqn" in s: return "QRDQN"
    if "a2c" in s: return "A2C"
    if "dqn" in s: return "DQN"
    if "ppo" in s: return "PPO"
    return "PPO"   # sensible default

UPLOADED_DIR = Path("/content/uploaded_models"); UPLOADED_DIR.mkdir(exist_ok=True)

# Keep any models registered by a previous run of this cell.
AGENT_REGISTRY = globals().get("AGENT_REGISTRY", {})   # label -> loaded model
AGENT_INFO     = globals().get("AGENT_INFO", {})       # label -> (algo, path)
# Defined so Sections 5-7 run even if you never executed the training section.
trained        = globals().get("trained", {})
train_history  = globals().get("train_history", {})

def _is_single_model(zip_path):
    """True for a real SB3 checkpoint (has a top-level 'data' member); False for
    an aggregate archive (which instead contains nested *.zip files)."""
    try:
        with zipfile.ZipFile(zip_path) as z:
            names = z.namelist()
    except zipfile.BadZipFile:
        return False
    return "data" in names and not any(n.endswith(".zip") for n in names)

def _register(label, algo, path):
    AGENT_REGISTRY[label] = _ALGO_CLASS[algo].load(str(path), device=_DEVICE)
    AGENT_INFO[label] = (algo, str(path))
    print(f"  loaded {label:<22} as {algo:<6} <- {Path(path).name}")

def _ingest(zip_path):
    zip_path = Path(zip_path)
    if _is_single_model(zip_path):
        algo  = ALGO_OVERRIDE.get(zip_path.name, _guess_algo(zip_path.name))
        label = LABEL_OVERRIDE.get(zip_path.name, zip_path.stem)
        _register(label, algo, zip_path); return
    # aggregate archive: unpack and register each <ALGO>/best_model.zip (or final.zip)
    out = UPLOADED_DIR / (zip_path.stem + "_unpacked")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out)
    found = 0
    for algo in _ALGO_CLASS:
        cand = out / algo / "best_model.zip"
        if not cand.exists(): cand = out / algo / "final.zip"
        if cand.exists():
            _register(algo, algo, cand); found += 1
    if not found:   # fall back: register every nested model zip we can find
        for cand in out.rglob("*.zip"):
            algo = ALGO_OVERRIDE.get(cand.name, _guess_algo(cand))
            _register(cand.parent.name or cand.stem, algo, cand); found += 1
    if not found:
        print("  !! no model checkpoints found inside", zip_path.name)

try:
    from google.colab import files  # type: ignore
    print("Upload a single SB3 checkpoint OR ths_trained_models.zip ...")
    _up = files.upload()
    _paths = []
    for _f in list(_up):
        if not _f.endswith(".zip"):
            print("  skip (not a .zip):", _f); continue
        _d = UPLOADED_DIR / _f
        if Path(_f).resolve() != _d.resolve(): shutil.move(_f, str(_d))
        _paths.append(_d)
except ImportError:
    print("Not on Colab - ingesting any .zip already in", UPLOADED_DIR)
    _paths = sorted(UPLOADED_DIR.glob("*.zip"))

for _p in _paths:
    _ingest(_p)

print("\nRegistered agents:", list(AGENT_REGISTRY))
print("These are benchmarked with NO retraining - you can skip Section 4.")


## 4 - Enhanced training

Improvements over a vanilla single-algorithm run:

* **Multi-cycle generalisation** - each episode randomly draws WLTC / FTP75 / US06, so the
  policy doesn't overfit one cycle.
* **Reward whitening** via `VecNormalize` (obs are already normalised inside the env) for a
  stable value function under the shaped charge-sustaining reward.
* **Linear-decay schedules** on learning rate (and PPO clip range) for clean convergence.
* **Per-algorithm tuned hyperparameters** (below) instead of library defaults.
* **`EvalCallback`** evaluating across all cycles and keeping the best checkpoint per algo.
* **Entropy bonus** (on-policy) / **decaying epsilon-greedy** (off-policy) to keep exploration alive.

Set `TOTAL_TIMESTEPS` below. ~200k is a quick demo (a few min/algo on GPU); use 1-2M for
publication-quality agents.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

from stable_baselines3 import PPO, A2C, DQN
from sb3_contrib import QRDQN
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize

from env.ths_env import THSEnv

ALL_CYCLES = ("WLTC", "FTP75", "US06")
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

# ---- knobs you can tune ---------------------------------------------------
TOTAL_TIMESTEPS = 200_000          # bump to 1_000_000+ for strong agents
ALGOS_TO_TRAIN  = ["PPO", "A2C", "DQN", "QRDQN"]   # drop any you don't want
N_ENVS_ONPOLICY = 4                # parallel rollout workers for PPO/A2C
# ---------------------------------------------------------------------------

MODELS_DIR = Path("/content/models"); MODELS_DIR.mkdir(exist_ok=True)
TB_DIR = Path("/content/tb_logs"); TB_DIR.mkdir(exist_ok=True)


class MultiCycleEnv(THSEnv):
    """Randomly pick a drive cycle on each reset for better generalisation."""
    def __init__(self, seed=0):
        self._rng = random.Random(seed)
        super().__init__(cycle=self._rng.choice(ALL_CYCLES))

    def reset(self, *, seed=None, options=None):
        self.cycle_name = self._rng.choice(ALL_CYCLES)
        self.cycle_path = (Path(self.cycle_path).parent / self.CYCLE_FILES[self.cycle_name])
        return super().reset(seed=seed, options=options)


def make_env(seed):
    def _init():
        e = MultiCycleEnv(seed=seed)
        e.reset(seed=seed)
        return Monitor(e)
    return _init


def make_vec(n_envs, seed):
    venv = DummyVecEnv([make_env(seed + i) for i in range(n_envs)])
    venv = VecMonitor(venv)
    return VecNormalize(venv, norm_obs=False, norm_reward=True, clip_reward=10.0)


def linear_schedule(initial, final_frac=0.1):
    final = initial * final_frac
    return lambda progress_remaining: final + (initial - final) * progress_remaining

### 4.1 - Per-algorithm builders

In [ ]:
def build_ppo(seed):
    venv = make_vec(N_ENVS_ONPOLICY, seed)
    model = PPO(
        "MlpPolicy", venv,
        learning_rate=linear_schedule(3e-4),
        n_steps=1024, batch_size=256, n_epochs=10,
        gamma=0.99, gae_lambda=0.95,
        clip_range=linear_schedule(0.2), ent_coef=0.01,
        policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128]), activation_fn=nn.Tanh),
        tensorboard_log=str(TB_DIR), seed=seed, device=DEVICE, verbose=0,
    )
    return model, venv

def build_a2c(seed):
    venv = make_vec(N_ENVS_ONPOLICY, seed)
    model = A2C(
        "MlpPolicy", venv,
        learning_rate=linear_schedule(7e-4),
        n_steps=20, gamma=0.99, gae_lambda=0.95,
        ent_coef=0.01, vf_coef=0.5,
        policy_kwargs=dict(net_arch=dict(pi=[128, 128], vf=[128, 128]), activation_fn=nn.Tanh),
        tensorboard_log=str(TB_DIR), seed=seed, device=DEVICE, verbose=0,
    )
    return model, venv

def build_dqn(seed):
    venv = make_vec(1, seed)   # off-policy: single env + replay buffer
    model = DQN(
        "MlpPolicy", venv,
        learning_rate=1e-3, buffer_size=200_000,
        learning_starts=5_000, batch_size=128, tau=1.0,
        gamma=0.99, train_freq=4, gradient_steps=1,
        target_update_interval=2_000,
        exploration_fraction=0.3, exploration_final_eps=0.02,
        policy_kwargs=dict(net_arch=[128, 128]),
        tensorboard_log=str(TB_DIR), seed=seed, device=DEVICE, verbose=0,
    )
    return model, venv

def build_qrdqn(seed):
    venv = make_vec(1, seed)
    model = QRDQN(
        "MlpPolicy", venv,
        learning_rate=1e-3, buffer_size=200_000,
        learning_starts=5_000, batch_size=128,
        gamma=0.99, train_freq=4, gradient_steps=1,
        target_update_interval=2_000,
        exploration_fraction=0.3, exploration_final_eps=0.02,
        policy_kwargs=dict(net_arch=[128, 128], n_quantiles=64),
        tensorboard_log=str(TB_DIR), seed=seed, device=DEVICE, verbose=0,
    )
    return model, venv

BUILDERS = {"PPO": build_ppo, "A2C": build_a2c, "DQN": build_dqn, "QRDQN": build_qrdqn}

### 4.2 - Train every selected algorithm

In [ ]:
import time

trained = {}          # name -> path to saved .zip
train_history = {}    # name -> (timesteps[], eval_reward[])

for name in ALGOS_TO_TRAIN:
    print(f"\n=== Training {name} for {TOTAL_TIMESTEPS:,} steps ===")
    set_random_seed(SEED)
    model, venv = BUILDERS[name](SEED)

    eval_env = make_vec(1, SEED + 10_000)
    eval_env.training = False
    eval_env.norm_reward = False
    algo_dir = MODELS_DIR / name
    eval_cb = EvalCallback(
        eval_env, best_model_save_path=str(algo_dir),
        log_path=str(algo_dir / "eval_logs"),
        eval_freq=max(25_000 // max(venv.num_envs, 1), 1),
        n_eval_episodes=3, deterministic=True, render=False,
    )

    t0 = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_cb,
                tb_log_name=f"{name.lower()}", progress_bar=True)
    dt = time.time() - t0

    final_path = algo_dir / "final.zip"
    model.save(final_path)
    # prefer the best checkpoint if EvalCallback saved one, else the final model
    best = algo_dir / "best_model.zip"
    trained[name] = str(best if best.exists() else final_path)

    npz = algo_dir / "eval_logs" / "evaluations.npz"
    if npz.exists():
        d = np.load(npz)
        train_history[name] = (d["timesteps"], d["results"].mean(axis=1))
    print(f"{name} done in {dt/60:.1f} min -> {trained[name]}")
    venv.close(); eval_env.close()

print("\nTrained:", list(trained))

## 5 - Benchmark: agents vs fixed modes vs rule baseline

Every contender runs through the same `THSEnv`. Fuel is read from the EMS accumulator
(`info['fuel_total_g']`), never integrated from the rate, so `dt` is handled correctly.
Agents predict on the raw env observation (the env already normalises obs, so `VecNormalize`
is *not* needed at eval - matching the project's benchmark convention).

In [ ]:
import pandas as pd

CYCLES = ("WLTC", "FTP75", "US06")
MODES = ("EV", "ECO", "NORMAL", "PWR")
ACTION_MAP = {"EV": 0, "ECO": 1, "NORMAL": 2, "PWR": 3}
SOC_TARGET = 60.0
GASOLINE_DENSITY_KG_L = 0.74
CO2_G_PER_G_FUEL = 3.09
FUEL_KWH_PER_G = 43.4 / 1000.0 / 3.6
DT = 1.0   # one profile sample == 1 s of cycle

# Optional pretrained PPO uploaded as best_model.zip
PRETRAINED = Path("/content/ths_project/best_model.zip")
if not PRETRAINED.exists():
    PRETRAINED = Path("/content/best_model.zip")


def rule_action(speed_ms, soc):
    kmh = speed_ms * 3.6
    if kmh < 5.0 and soc >= 0.45: return 0   # EV
    if kmh < 15.0: return 1                   # ECO
    if kmh < 80.0: return 2                   # NORMAL
    return 3                                  # PWR


def run_episode(cycle, *, model=None, fixed_action=None, rule=False):
    env = THSEnv(cycle=cycle)
    obs, _ = env.reset(seed=0)
    rows, done = [], False
    while not done:
        if rule:
            action = rule_action(float(env.speed),
                                  float(env.ems.state.soc) if env.ems else 0.6)
        elif model is not None:
            action = int(model.predict(obs, deterministic=True)[0])
        else:
            action = fixed_action
        obs, _r, term, trunc, info = env.step(action)
        done = term or trunc
        rows.append({
            "soc_pct": float(info["soc_pct"]),
            "fuel_total_g": float(info["fuel_total_g"]),
            "fuel_rate_gs": float(info["fuel_rate_gs"]),
            "p_batt_kw": float(info["p_batt_kw"]),
            "speed_ms": float(info["target_speed_ms"]),
            "actual_speed_ms": float(env.speed),
            "ice_on": bool(info["ice_on"]),
            "action": action,
        })
    return pd.DataFrame(rows)


def kpis(df, cycle, label):
    soc = df["soc_pct"].to_numpy(float)
    p_batt = df["p_batt_kw"].to_numpy(float)
    ice = df["ice_on"].to_numpy(bool)
    fuel = float(df["fuel_total_g"].iloc[-1])
    dist_km = float(np.sum(df["speed_ms"].to_numpy() * DT) / 1000.0)
    liters = (fuel / 1000.0) / GASOLINE_DENSITY_KG_L
    l100 = liters / dist_km * 100.0 if dist_km > 0 else np.nan
    regen_kj = float(np.sum(np.where(~ice, np.maximum(0, -p_batt), 0) * 1000.0 * DT) / 1000.0)
    ice_starts = int(np.sum((~ice[:-1]) & ice[1:])) if len(ice) > 1 else int(ice[0])
    spd_err = (df["actual_speed_ms"].to_numpy() - df["speed_ms"].to_numpy()) * 3.6
    return {
        "cycle": cycle, "contender": label,
        "total_fuel_g": round(fuel, 1),
        "fuel_g_per_km": round(fuel / dist_km, 3) if dist_km > 0 else np.nan,
        "fuel_l_per_100km": round(l100, 2),
        "co2_g": round(fuel * CO2_G_PER_G_FUEL, 1),
        "energy_kwh": round(fuel * FUEL_KWH_PER_G, 3),
        "soc_final_pct": round(float(soc[-1]), 1),
        "soc_rmse": round(float(np.sqrt(np.mean((soc - SOC_TARGET) ** 2))), 3),
        "soc_min_pct": round(float(soc.min()), 1),
        "ice_on_pct": round(float(ice.mean()) * 100, 1),
        "ice_starts": ice_starts,
        "regen_kj": round(regen_kj, 1),
        "speed_rmse_kmh": round(float(np.sqrt(np.mean(spd_err ** 2))), 2),
    }

In [ ]:
# Assemble the contender list: fixed modes + rule baseline + every trained agent (+ pretrained)
# DEVICE may be undefined if you skipped Section 4 (training) - fall back.
DEVICE = globals().get("DEVICE", "cuda" if __import__("torch").cuda.is_available() else "cpu")

agents = {}
for name, path in globals().get("trained", {}).items():
    cls = {"PPO": PPO, "A2C": A2C, "DQN": DQN, "QRDQN": QRDQN}[name]
    agents[name] = cls.load(path, device=DEVICE)
# Models you uploaded in 'Use models you already trained' - no retraining needed.
agents.update(globals().get("AGENT_REGISTRY", {}))
if PRETRAINED.exists():
    agents["PPO-pretrained"] = PPO.load(str(PRETRAINED), device=DEVICE)
    print("Loaded pretrained model:", PRETRAINED)

rows, traces = [], {}
for cycle in CYCLES:
    print(f"\n--- {cycle} ---")
    for mode in MODES:
        df = run_episode(cycle, fixed_action=ACTION_MAP[mode])
        traces[(cycle, mode)] = df
        rows.append(kpis(df, cycle, mode))
        print(f"  Fixed {mode:<7} fuel={rows[-1]['total_fuel_g']:7.1f}g  SOC_f={rows[-1]['soc_final_pct']:5.1f}%")
    df = run_episode(cycle, rule=True)
    traces[(cycle, "Rule")] = df
    rows.append(kpis(df, cycle, "Rule"))
    print(f"  Rule-based   fuel={rows[-1]['total_fuel_g']:7.1f}g  SOC_f={rows[-1]['soc_final_pct']:5.1f}%")
    for name, model in agents.items():
        df = run_episode(cycle, model=model)
        traces[(cycle, name)] = df
        rows.append(kpis(df, cycle, name))
        print(f"  {name:<12} fuel={rows[-1]['total_fuel_g']:7.1f}g  SOC_f={rows[-1]['soc_final_pct']:5.1f}%")

bench = pd.DataFrame(rows)
bench.to_csv("/content/benchmark_kpis.csv", index=False)
CONTENDERS = list(MODES) + ["Rule"] + list(agents.keys())
print("\nContenders:", CONTENDERS)
bench.head()

## 6 - Leaderboard: charge-balanced fuel economy

In [ ]:
# Rank by mean fuel/km across cycles, but only count SOC-fair runs (final SOC within
# 5 points of the 60% target - otherwise a contender 'wins' by draining the battery).
fair = bench.copy()
fair["soc_fair"] = (fair["soc_final_pct"] - SOC_TARGET).abs() <= 5.0

board = (fair.groupby("contender")
              .agg(fuel_g_per_km=("fuel_g_per_km", "mean"),
                   co2_g=("co2_g", "mean"),
                   soc_rmse=("soc_rmse", "mean"),
                   ice_on_pct=("ice_on_pct", "mean"),
                   soc_fair_runs=("soc_fair", "sum"))
              .sort_values("fuel_g_per_km"))
board["rank"] = range(1, len(board) + 1)
print("Lower fuel_g_per_km is better. soc_fair_runs = # of cycles ending near 60% SOC (max 3).")
board

## 7 - Plots

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"EV":"#4e79a7","ECO":"#59a14f","NORMAL":"#f28e2b","PWR":"#e15759","Rule":"#9c755f"}
AGENT_PALETTE = ["#000000","#7b3294","#1b9e77","#d95f02","#666666"]
for i, name in enumerate(list(agents.keys())):
    COLORS[name] = AGENT_PALETTE[i % len(AGENT_PALETTE)]


def grouped_bar(ax, value_col, ylabel, title):
    x = np.arange(len(CYCLES)); width = 0.8 / len(CONTENDERS)
    for i, name in enumerate(CONTENDERS):
        vals = [float(bench[(bench.cycle == c) & (bench.contender == name)][value_col].iloc[0]) for c in CYCLES]
        off = (i - (len(CONTENDERS) - 1) / 2) * width
        is_agent = name in agents
        ax.bar(x + off, vals, width, label=name, color=COLORS.get(name, "#888"),
               alpha=0.95 if is_agent else 0.7, edgecolor="black" if is_agent else "none",
               linewidth=1.1 if is_agent else 0, zorder=3 if is_agent else 2)
    ax.set_xticks(x); ax.set_xticklabels(CYCLES)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=0.3)

fig, ax = plt.subplots(figsize=(13, 6))
grouped_bar(ax, "fuel_g_per_km", "Fuel economy (g/km)", "Fuel Economy - RL agents vs fixed modes vs rule")
ax.legend(ncol=len(CONTENDERS), fontsize=8)
plt.tight_layout(); plt.savefig("/content/bench_fuel_per_km.png", dpi=200); plt.show()

In [ ]:
# Fuel + CO2 + SOC RMSE side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
grouped_bar(axes[0], "total_fuel_g", "Total fuel (g)", "Total fuel")
grouped_bar(axes[1], "co2_g", "CO2 (g)", "Tailpipe CO2")
grouped_bar(axes[2], "soc_rmse", "SOC RMSE (%)", "Charge-sustaining error (lower=better)")
axes[0].legend(ncol=2, fontsize=7)
plt.tight_layout(); plt.savefig("/content/bench_fuel_co2_soc.png", dpi=200); plt.show()

In [ ]:
# SOC trajectories per cycle: agents (solid) vs fixed modes (dashed)
fig, axes = plt.subplots(len(CYCLES), 1, figsize=(12, 3.2 * len(CYCLES)))
axes = np.atleast_1d(axes)
for ax, cycle in zip(axes, CYCLES):
    for mode in MODES:
        df = traces[(cycle, mode)]
        ax.plot(df.index, df.soc_pct, color=COLORS[mode], lw=0.9, ls="--", alpha=0.6, label=f"Fixed {mode}")
    for name in agents:
        df = traces[(cycle, name)]
        ax.plot(df.index, df.soc_pct, color=COLORS[name], lw=1.8, label=name, zorder=5)
    ax.axhline(60, color="gray", ls=":", lw=0.8)
    ax.axhline(40, color="red", ls=":", lw=0.8, alpha=0.5)
    ax.set_title(cycle); ax.set_ylabel("SOC (%)"); ax.grid(alpha=0.2)
    ax.legend(fontsize=7, ncol=4, loc="upper right")
axes[-1].set_xlabel("Step")
fig.suptitle("SOC trajectory: RL agents vs fixed modes", fontsize=13)
plt.tight_layout(); plt.savefig("/content/bench_soc_traces.png", dpi=200); plt.show()

In [ ]:
# Agent mode-selection distribution (how each RL policy uses the 4 modes)
if agents:
    fig, axes = plt.subplots(len(agents), len(CYCLES), figsize=(4 * len(CYCLES), 3 * len(agents)),
                             squeeze=False)
    bar_colors = [COLORS[m] for m in MODES]
    for r, name in enumerate(agents):
        for c, cycle in enumerate(CYCLES):
            ax = axes[r][c]
            counts = traces[(cycle, name)]["action"].value_counts().reindex(range(4), fill_value=0)
            pct = counts / counts.sum() * 100
            ax.bar(MODES, pct.values, color=bar_colors)
            ax.set_ylim(0, 105)
            if c == 0: ax.set_ylabel(f"{name}\n% of steps", fontsize=9)
            if r == 0: ax.set_title(cycle)
    fig.suptitle("RL agent mode-selection distribution", fontsize=13)
    plt.tight_layout(); plt.savefig("/content/bench_action_dist.png", dpi=200); plt.show()

In [ ]:
# Training curves (eval reward vs timesteps) for each algorithm
if train_history:
    plt.figure(figsize=(11, 6))
    for name, (ts, rew) in train_history.items():
        plt.plot(ts, rew, marker="o", ms=3, label=name, color=COLORS.get(name, None))
    plt.xlabel("Timesteps"); plt.ylabel("Mean eval episode reward")
    plt.title("Training progress - eval reward across all cycles")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("/content/training_curves.png", dpi=200); plt.show()
else:
    print("No eval logs captured (increase TOTAL_TIMESTEPS so EvalCallback fires).")

In [ ]:
# Normalised radar per cycle (outer = better on every axis)
def radar(out="/content/bench_radar.png"):
    spec = [("total_fuel_g","Fuel",True), ("fuel_g_per_km","Fuel/km",True),
            ("soc_rmse","SOC dev",True), ("ice_on_pct","ICE use",True),
            ("soc_min_pct","SOC floor",False), ("regen_kj","Regen",False)]
    labels = [s[1] for s in spec]
    ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]
    fig, axes = plt.subplots(1, len(CYCLES), figsize=(5.2*len(CYCLES), 5.2),
                             subplot_kw={"polar": True}); axes = np.atleast_1d(axes)
    for ax, cycle in zip(axes, CYCLES):
        sub = bench[bench.cycle == cycle]
        bounds = {c:(float(sub[c].min()), float(sub[c].max())) for c,_,_ in spec}
        for name in CONTENDERS:
            row = sub[sub.contender == name].iloc[0]; scores = []
            for c,_,low in spec:
                lo, hi = bounds[c]; span = hi - lo
                nrm = 0.5 if span == 0 else (float(row[c]) - lo) / span
                scores.append(1 - nrm if low else nrm)
            scores += scores[:1]; is_agent = name in agents
            ax.plot(ang, scores, color=COLORS.get(name, "#888"),
                    lw=2.2 if is_agent else 1.2, ls="-" if is_agent else "--", label=name)
            if is_agent: ax.fill(ang, scores, color=COLORS.get(name), alpha=0.08)
        ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylim(0, 1); ax.set_yticklabels([]); ax.set_title(cycle, pad=16)
    axes[-1].legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=8)
    fig.suptitle("Normalised performance radar (outer = better)", fontsize=13)
    plt.tight_layout(); plt.savefig(out, dpi=200); plt.show()

radar()

## 8 - Download results

In [ ]:
import shutil
shutil.make_archive("/content/ths_trained_models", "zip", str(MODELS_DIR))

try:
    from google.colab import files  # type: ignore
    files.download("/content/benchmark_kpis.csv")
    files.download("/content/ths_trained_models.zip")
    print("Downloading benchmark_kpis.csv and trained models. PNG figures are in /content.")
except ImportError:
    print("Saved to /content: benchmark_kpis.csv, *.png, ths_trained_models.zip")

## 9 - GENERAL all-regime benchmark (`eval/benchmark_general.py`)

Runs every contender through the **GENERAL** cycle - one ~1327 s / ~24 km synthetic
route that chains cold-start idle, urban stop-and-go, congestion, rural road, a +-8%
mountain pass, motorway cruise and high-speed traffic, with a real road-grade profile.
This reuses the project's own `eval/benchmark_general.py` so the metrics match exactly:

1. **CO2** (g and g/km)  2. **Total energy** (fuel + electricity, kWh and Wh/km)
3. **SOC trajectory**  4. **Fuel** (g, g/km, L/100km, mpg)
5. **Battery-life prediction** (throughput aging -> km / years to EOL)
6. **Agent decisions** - which mode (EV/ECO/NORMAL/PWR) each agent picks on every road part.

Fixed modes plus **every model you registered above** are compared - no retraining.


In [ ]:
# === Section 9: GENERAL all-regime benchmark ================================
import importlib, sys
from pathlib import Path
import numpy as np, pandas as pd

# benchmark_general.py ships in the bundle under eval/ - import it as a module.
EVAL_DIR = Path("/content/ths_project/eval")
if str(EVAL_DIR) not in sys.path: sys.path.insert(0, str(EVAL_DIR))
import benchmark_general as bg
importlib.reload(bg)

KM_PER_YEAR = 15000.0   # annual-mileage assumption for the battery-life-in-years KPI

# Agents to benchmark = everything you uploaded; fall back to a stray best_model.zip.
gen_agents = dict(globals().get("AGENT_REGISTRY", {}))
if not gen_agents:
    _pre = Path("/content/ths_project/best_model.zip")
    if not _pre.exists(): _pre = Path("/content/best_model.zip")
    if _pre.exists():
        from stable_baselines3 import PPO
        gen_agents["PPO-pretrained"] = PPO.load(str(_pre))
assert gen_agents, "No agents registered - run the 'Use models you already trained' cell first."

GEN_CONTENDERS = list(bg.MODES) + list(gen_agents)
gen_rows, gen_traces = [], {}
print(f"--- GENERAL  ({len(gen_agents)} agent(s) + 4 fixed modes) ---")
for _mode in bg.MODES:
    _df = bg.run_episode(bg.CYCLE, model=None, fixed_action=bg.ACTION_MAP[_mode])
    gen_traces[_mode] = _df
    gen_rows.append(bg.compute_kpis(_df, bg.CYCLE, _mode, KM_PER_YEAR))
    _k = gen_rows[-1]
    print(f"  Fixed {_mode:<7} fuel={_k['total_fuel_g']:7.1f}g  CO2={_k['co2_g']:7.1f}g  SOCf={_k['soc_final_pct']:5.1f}%  life={_k['batt_life_km']:.0f}km")
for _label, _model in gen_agents.items():
    _df = bg.run_episode(bg.CYCLE, model=_model, fixed_action=None)
    gen_traces[_label] = _df
    gen_rows.append(bg.compute_kpis(_df, bg.CYCLE, _label, KM_PER_YEAR))
    _k = gen_rows[-1]
    print(f"  {_label:<13} fuel={_k['total_fuel_g']:7.1f}g  CO2={_k['co2_g']:7.1f}g  SOCf={_k['soc_final_pct']:5.1f}%  life={_k['batt_life_km']:.0f}km")

gen_kpis = pd.DataFrame(gen_rows)
gen_kpis.to_csv("/content/general_kpis.csv", index=False)
print("\nSaved /content/general_kpis.csv")
gen_kpis


In [ ]:
# GENERAL: multi-agent comparison bars + SOC trajectories
import matplotlib.pyplot as plt, numpy as np

GEN_COLORS = dict(bg.COLORS)   # EV/ECO/NORMAL/PWR (+ a default 'Agent' black)
_palette = ["#000000", "#7b3294", "#1b9e77", "#d95f02", "#666666", "#e7298a"]
for _i, _label in enumerate(gen_agents):
    GEN_COLORS[_label] = _palette[_i % len(_palette)]

def _gbar(ax, col, ylabel, title, fmt="{:.1f}"):
    vals = [float(gen_kpis[gen_kpis.contender == c][col].iloc[0]) for c in GEN_CONTENDERS]
    is_a = [c in gen_agents for c in GEN_CONTENDERS]
    bars = ax.bar(GEN_CONTENDERS, vals,
                  color=[GEN_COLORS.get(c, "#888") for c in GEN_CONTENDERS], alpha=0.9,
                  edgecolor=["black" if a else "none" for a in is_a],
                  linewidth=[1.3 if a else 0 for a in is_a])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt.format(v),
                ha="center", va="bottom", fontsize=7)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
_gbar(axes[0, 0], "total_fuel_g",     "Fuel (g)",  "Total fuel")
_gbar(axes[0, 1], "co2_g",            "CO2 (g)",   "Tailpipe CO2")
_gbar(axes[1, 0], "energy_wh_per_km", "Wh/km",     "Total energy per km")
_gbar(axes[1, 1], "batt_life_km",     "Life (km)", "Predicted battery life", "{:.0f}")
fig.suptitle("GENERAL all-regime cycle - fixed modes vs your trained agents", fontsize=14)
plt.tight_layout(); plt.savefig("/content/general_compare.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(13, 5))
for _mode in bg.MODES:
    _d = gen_traces[_mode]
    ax.plot(_d.time_s, _d.soc_pct, color=GEN_COLORS[_mode], ls="--", lw=0.9, alpha=0.7, label=f"Fixed {_mode}")
for _label in gen_agents:
    _d = gen_traces[_label]
    ax.plot(_d.time_s, _d.soc_pct, color=GEN_COLORS[_label], lw=1.9, label=_label, zorder=5)
ax.axhline(60, color="gray", ls=":", lw=0.8); ax.axhline(40, color="red", ls=":", lw=0.8, alpha=0.5)
ax.set_xlabel("Time (s)"); ax.set_ylabel("SOC (%)"); ax.set_title("SOC trajectory - GENERAL cycle")
ax.grid(alpha=0.2); ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout(); plt.savefig("/content/general_soc_trace.png", dpi=200); plt.show()


In [ ]:
# GENERAL: per-agent decision timeline + per-road-part mode share (one set per agent)
from IPython.display import Image, display

phases = bg.load_phases()
for _label in gen_agents:
    _adf = gen_traces[_label]
    _safe = _label.replace("/", "-").replace(" ", "_")
    # speed shaded by the mode the agent chose, with a road-grade strip
    _dec = bg.save_decision_timeline(_adf, phases, bg.FIGURE_DIR / f"general_decisions_{_safe}.png")
    # mode mix per road phase (table + stacked bars)
    _share = bg.compute_phase_mode_share(_adf, phases)
    _share.to_csv(f"/content/general_phase_mode_share_{_safe}.csv", index=False)
    _shr = bg.save_phase_mode_share(_share, bg.FIGURE_DIR / f"general_phaseshare_{_safe}.png")
    print(f"\n=== {_label} ===")
    display(Image(str(_dec)))
    display(Image(str(_shr)))


In [ ]:
# Download the GENERAL results (CSVs + figures)
import shutil
shutil.make_archive("/content/general_figures", "zip", str(bg.FIGURE_DIR))
try:
    from google.colab import files  # type: ignore
    files.download("/content/general_kpis.csv")
    files.download("/content/general_compare.png")
    files.download("/content/general_soc_trace.png")
    files.download("/content/general_figures.zip")
    print("Downloaded GENERAL KPIs, figures and the figure bundle.")
except ImportError:
    print("Saved to /content: general_kpis.csv, general_*.png, general_figures.zip")
